In [1]:
from dask.distributed import LocalCluster, Client
cluster = LocalCluster()
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 16
Total threads: 128,Total memory: 0.98 TiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:40969,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:35997,Total threads: 8
Dashboard: http://127.0.0.1:42937/status,Memory: 62.93 GiB
Nanny: tcp://127.0.0.1:38523,


### Load MITgcm Output from Assimilations

In [2]:
foldername = '/home/edavenport/analysis/vel-assim-manuscript/assimilation_results/'

In [3]:
import matplotlib.pyplot as plt
plt.rcParams['font.size'] = 13
import cmocean.cm as cmo
import xarray as xr
from open_tpose import tpose2012to2013
import numpy as np
import xarray as xr

prefix = ['diag_state']
ds_tpose_noTAO = tpose2012to2013(prefix)

ds_tpose_noTAO['XC'] = ds_tpose_noTAO.XC.astype(float)
ds_tpose_noTAO['YC'] = ds_tpose_noTAO.YC.astype(float)
ds_tpose_noTAO['Z'] = ds_tpose_noTAO.Z.astype(float)
ds_tpose_noTAO['XG'] = ds_tpose_noTAO.XG.astype(float)
ds_tpose_noTAO['YG'] = ds_tpose_noTAO.YG.astype(float)

mar2013/diags_daily/
may2013/diags_daily/
jul2013/diags_daily/
sep2013/diags_daily/
nov2013/diags_daily/
Days in 2012-2013: (should be 731)
731


In [4]:
from xmitgcm import open_mdsdataset
data_dir = '/data/SO3/edavenport/tpose6/sep2012/velocity_assim/run_iter22/'
grid_dir = '/data/SO6/TPOSE_diags/tpose6/grid_6/'

offset = 10
num_diags = 30+31+offset #sep, oct + 10 days
itPerFile = 72 # 1 day
intervals = range(itPerFile,itPerFile*num_diags,itPerFile)

ds = open_mdsdataset(data_dir=data_dir,grid_dir=grid_dir,iters=intervals,prefix=prefix,ref_date='2012-09-01',delta_t=1200)

num_diags = 30+31+offset# nov, dec (starting from nov 10)
itPerFile = 72 # 1 day
intervals = range(itPerFile*offset,itPerFile*num_diags,itPerFile)

data_dir = '/data/SO3/edavenport/tpose6/nov2012/run_iter20/'
ds_new = open_mdsdataset(data_dir=data_dir,grid_dir=grid_dir,iters=intervals,prefix=prefix,ref_date='2012-11-01',delta_t=1200)

ds_tpose_TAO = xr.concat([ds,ds_new],dim='time')

num_diags = 31+28+offset # jan, feb, (starting from jan 10)
itPerFile = 72 # 1 day
intervals = range(itPerFile*offset,itPerFile*num_diags,itPerFile)

data_dir = '/data/SO3/edavenport/tpose6/jan2013/run_iter14/'
ds_new = open_mdsdataset(data_dir=data_dir,grid_dir=grid_dir,iters=intervals,prefix=prefix,ref_date='2013-01-01',delta_t=1200)

ds_tpose_TAO = xr.concat([ds_tpose_TAO,ds_new],dim='time')

num_diags = 31+30+31+30 # mar, apr, may, june (starting from jan 10)
itPerFile = 72 # 1 day
intervals = range(itPerFile*offset,itPerFile*num_diags,itPerFile)

data_dir = '/data/SO3/edavenport/tpose6/mar2013/run_iter16/'
ds_new = open_mdsdataset(data_dir=data_dir,grid_dir=grid_dir,iters=intervals,prefix=prefix,ref_date='2013-03-01',delta_t=1200)

ds_tpose_TAO = xr.concat([ds_tpose_TAO,ds_new],dim='time')

ds_tpose_TAO['XC'] = ds_tpose_TAO.XC.astype(float)
ds_tpose_TAO['YC'] = ds_tpose_TAO.YC.astype(float)
ds_tpose_TAO['Z'] = ds_tpose_TAO.Z.astype(float)
ds_tpose_TAO['XG'] = ds_tpose_TAO.XG.astype(float)
ds_tpose_TAO['YG'] = ds_tpose_TAO.YG.astype(float)

### Load TAO U and V

In [21]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
plt.rcParams['font.size'] = 13

fig = plt.figure(figsize=(14, 21))

gs = gridspec.GridSpec(
    12, 2,
    height_ratios=[1, 1, 1, 0.06, 1, 1, 1, 0.06, 1, 1, 1, 0.06],
    hspace=0.65,
    wspace=0.15
)

<Figure size 1400x2000 with 0 Axes>

In [22]:
lon_str = '170'
rows = np.arange(0,4,1)

In [23]:
TAO_ADCP_file = '/data/SO3/edavenport/TAO_2012to2016_daily/ADCP_2012to2016_0N' + lon_str + 'W_daily.cdf' 
dsTAO = xr.open_dataset(TAO_ADCP_file)
dsTAO['depth'] = -1*dsTAO.depth
dsTAO = dsTAO.sel(time=slice('2012-09-01','2013-06-30')) # subset to sep 2012, apr 2013
dsTAO['u_1205'] = dsTAO.u_1205/100 # convert from cm/s to m/s
dsTAO['v_1206'] = dsTAO.v_1206/100 # convert from cm/s to m/s

# rename TAO coords/dims so that depth is Z, lat is YC, lon is XC
dsTAO = dsTAO.rename({'depth':'Z','lat':'YC','lon':'XG'})
tpose_U_noTAO = ds_tpose_noTAO.UVEL.interp_like(dsTAO.u_1205).compute().squeeze()
tpose_U_TAO = ds_tpose_TAO.UVEL.interp_like(dsTAO.u_1205).compute().squeeze()

In [24]:
# Create axes with shared x within each column
ax = np.empty((3, 2), dtype=object)

ax[0,0] = fig.add_subplot(gs[rows[0],0])
ax[1,0] = fig.add_subplot(gs[rows[1],0], sharex=ax[0,0])
ax[2,0] = fig.add_subplot(gs[rows[2],0], sharex=ax[0,0])

ax[0,1] = fig.add_subplot(gs[rows[0],1])
ax[1,1] = fig.add_subplot(gs[rows[1],1], sharex=ax[0,1])
ax[2,1] = fig.add_subplot(gs[rows[2],1], sharex=ax[0,1])

vmin, vmax = -1.35, 1.35

# ----- Left column -----
mappable = dsTAO.u_1205.plot(
    ax=ax[0,0], cmap=cmo.balance,
    vmin=vmin, vmax=vmax,
    x='time', y='Z',
    add_colorbar=False
)
tpose_U_noTAO.plot(
    ax=ax[1,0], cmap=cmo.balance,
    vmin=vmin, vmax=vmax,
    x='time', y='Z',
    add_colorbar=False
)
(tpose_U_noTAO - dsTAO.u_1205).plot(
    ax=ax[2,0], cmap=cmo.balance,
    vmin=vmin, vmax=vmax,
    x='time', y='Z',
    add_colorbar=False
)

# ----- Right column -----
dsTAO.u_1205.plot(
    ax=ax[0,1], cmap=cmo.balance,
    vmin=vmin, vmax=vmax,
    x='time', y='Z',
    add_colorbar=False
)
tpose_U_TAO.plot(
    ax=ax[1,1], cmap=cmo.balance,
    vmin=vmin, vmax=vmax,
    x='time', y='Z',
    add_colorbar=False
)
(tpose_U_TAO - dsTAO.u_1205).plot(
    ax=ax[2,1], cmap=cmo.balance,
    vmin=vmin, vmax=vmax,
    x='time', y='Z',
    add_colorbar=False
)

title = '0N, ' + lon_str + 'W'
# Remove titles everywhere
for i in range(3):
    for j in range(2):
        ax[i,j].set_ylim(-250, -20)
        ax[i,j].set_xlabel('')
        ax[i,j].text(
        0.99, 0.16, title,
        transform=ax[i, j].transAxes,
        ha='right', va='top',
        fontsize=14,
        bbox=dict(
            boxstyle='round,pad=0.25',
            facecolor='white',
            edgecolor='k',
            linewidth=0.5,
            alpha=1.0
        )
    )

# Remove y-labels on right column
for i in range(3):
    ax[i,1].set_ylabel('')
    ax[i,0].tick_params(labelbottom=False)
    ax[i,1].tick_params(labelbottom=False)

ax[2,0].tick_params(labelbottom=True)
ax[2,1].tick_params(labelbottom=True)

ax[0,0].set_title('Velocity Observations')
ax[1,0].set_title('TPOSE.6')
ax[2,0].set_title('Model-Data Difference')
ax[0,1].set_title('Velocity Observations')
ax[1,1].set_title('TPOSE.6-TAO')
ax[2,1].set_title('Model-Data Difference')

# ----- Colorbar -----
cax = fig.add_subplot(gs[rows[3], :])
cbar = fig.colorbar(mappable, cax=cax, orientation='horizontal')
cbar.set_label('U (m/s')

In [25]:
plt.tight_layout()
fig.savefig(foldername+'zonal_velocity.png')

<Figure size 640x480 with 0 Axes>

## Meridional Velocity

In [56]:
fig = plt.figure(figsize=(16, 20))

gs = gridspec.GridSpec(
    10, 2,
    height_ratios=[1, 1, 1, 1, 1, 1, 1, 1, 1, 0.06],
    hspace=0.65,
    wspace=0.15
)

<Figure size 1600x2000 with 0 Axes>

In [63]:
lon_str = '110'
rows = np.arange(6,9,1)

In [64]:
TAO_ADCP_file = '/data/SO3/edavenport/TAO_2012to2016_daily/ADCP_2012to2016_0N' + lon_str + 'W_daily.cdf' 
dsTAO = xr.open_dataset(TAO_ADCP_file)
dsTAO['depth'] = -1*dsTAO.depth
dsTAO = dsTAO.sel(time=slice('2012-09-01','2013-06-30')) # subset to sep 2012, apr 2013
dsTAO['u_1205'] = dsTAO.u_1205/100 # convert from cm/s to m/s
dsTAO['v_1206'] = dsTAO.v_1206/100 # convert from cm/s to m/s

dsTAO = dsTAO.rename({'depth':'Z','lat':'YG','lon':'XC'})
tpose_V_noTAO = ds_tpose_noTAO.VVEL.interp_like(dsTAO.v_1206).compute().squeeze()
tpose_V_TAO = ds_tpose_TAO.VVEL.interp_like(dsTAO.v_1206).compute().squeeze()

In [65]:
# Create axes with shared x within each column
ax = np.empty((3, 2), dtype=object)

ax[0,0] = fig.add_subplot(gs[rows[0],0])
ax[1,0] = fig.add_subplot(gs[rows[1],0], sharex=ax[0,0])
ax[2,0] = fig.add_subplot(gs[rows[2],0], sharex=ax[0,0])

ax[0,1] = fig.add_subplot(gs[rows[0],1])
ax[1,1] = fig.add_subplot(gs[rows[1],1], sharex=ax[0,1])
ax[2,1] = fig.add_subplot(gs[rows[2],1], sharex=ax[0,1])

vmin, vmax = -0.6, 0.6

# ----- Left column -----
mappable = dsTAO.v_1206.plot(
    ax=ax[0,0], cmap=cmo.balance,
    vmin=vmin, vmax=vmax,
    x='time', y='Z',
    add_colorbar=False
)
tpose_V_noTAO.plot(
    ax=ax[1,0], cmap=cmo.balance,
    vmin=vmin, vmax=vmax,
    x='time', y='Z',
    add_colorbar=False
)
(tpose_V_noTAO - dsTAO.v_1206).plot(
    ax=ax[2,0], cmap=cmo.balance,
    vmin=vmin, vmax=vmax,
    x='time', y='Z',
    add_colorbar=False
)

# ----- Right column -----
dsTAO.v_1206.plot(
    ax=ax[0,1], cmap=cmo.balance,
    vmin=vmin, vmax=vmax,
    x='time', y='Z',
    add_colorbar=False
)
tpose_V_TAO.plot(
    ax=ax[1,1], cmap=cmo.balance,
    vmin=vmin, vmax=vmax,
    x='time', y='Z',
    add_colorbar=False
)
(tpose_V_TAO - dsTAO.v_1206).plot(
    ax=ax[2,1], cmap=cmo.balance,
    vmin=vmin, vmax=vmax,
    x='time', y='Z',
    add_colorbar=False
)

title = '0N, ' + lon_str + 'W'
# Remove titles everywhere
for i in range(3):
    for j in range(2):
        ax[i,j].set_ylim(-250, -20)
        ax[i,j].set_xlabel('')
        ax[i,j].text(
        0.99, 0.16, title,
        transform=ax[i, j].transAxes,
        ha='right', va='top',
        fontsize=14,
        bbox=dict(
            boxstyle='round,pad=0.25',
            facecolor='white',
            edgecolor='k',
            linewidth=0.5,
            alpha=1.0
        )
    )

# Remove y-labels on right column
for i in range(3):
    ax[i,1].set_ylabel('')
    ax[i,0].tick_params(labelbottom=False)
    ax[i,1].tick_params(labelbottom=False)

ax[0,0].set_title('Velocity Observations')
ax[1,0].set_title('TPOSE.6')
ax[2,0].set_title('Model-Data Difference')
ax[0,1].set_title('Velocity Observations')
ax[1,1].set_title('TPOSE.6-TAO')
ax[2,1].set_title('Model-Data Difference')


Text(0.5, 1.0, 'Model-Data Difference')

In [66]:
ax[2,0].tick_params(labelbottom=True)
ax[2,1].tick_params(labelbottom=True)

# ----- Colorbar -----
cax = fig.add_subplot(gs[9, :])
cbar = fig.colorbar(mappable, cax=cax, orientation='horizontal')
cbar.set_label('V (m/s)')

plt.tight_layout()
fig.savefig(foldername+'meridional_velocity.png')

<Figure size 640x480 with 0 Axes>

In [67]:
client.shutdown()
cluster.close()
client.close()